# Learned detector: training and comparison only

This is the **only** notebook in the project, and it does exactly one job: train and compare
learned detection models on a dataset that `experiments/fault_detection/evaluate.py` has
already validated.

It deliberately does **not** simulate, generate workloads, inject faults, clean telemetry,
build features, derive labels, or orchestrate experiments. Those are plain-Python stages:

```
run_simulation.py  ->  inject_faults.py  ->  evaluate.py  ->  windows.csv
                                                              |
                                                              +-> this notebook
```

Input contract: `windows.csv` written by `evaluate.py` (one row per window x instance, with
features, `label_kind`, `pair_id` and `run_id`). Ground truth never enters the feature matrix.

**Scope of any number produced here:** synthetic faults in a simulator. This validates the
detection machinery, not real-hardware fault detection.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

# Repo root = two levels up from notebooks/ (fallback: cwd).
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) != "notebooks":
    REPO_ROOT = os.getcwd()
sys.path.insert(0, REPO_ROOT)

from experiments.fault_detection.evaluate import (  # noqa: E402
    FEATURE_COLUMNS,
    POSITIVE_KINDS,
    bootstrap_over_runs,
    prf_counts,
    prf_from_counts,
)

REPORT_DIR = os.path.join(REPO_ROOT, "experiments", "report", "review_min")
SEED = 42
MIN_RUNS = 6          # below this, a learned model is not defensible
print("report dir:", REPORT_DIR)

In [ ]:
windows_path = os.path.join(REPORT_DIR, "windows.csv")
metrics_path = os.path.join(REPORT_DIR, "metrics.csv")
if not os.path.exists(windows_path):
    raise SystemExit(
        "No validated dataset at %s.\n" % windows_path
        + "Run the pipeline first:\n"
        + "  python -m experiments.fault_detection.inject_faults "
          "--matrix experiments/fault_detection/matrix.yaml "
          "--profile review_min --out experiments/out/review_min "
          "--seeds 1,2,3,4,5\n"
        + "  python -m experiments.fault_detection.evaluate "
          "--data experiments/out/review_min "
          "--report experiments/report/review_min\n"
        + "This notebook never fabricates or repairs a dataset.")

df = pd.read_csv(windows_path)
print(df.shape)
print(df["label_kind"].value_counts())
print("runs:", df["run_id"].nunique(), " pairs:", df["pair_id"].nunique())

In [ ]:
# Only root_cause vs unaffected are scoreable; downstream/recovering/neutral
# are reported separately by evaluate.py and excluded here for the same reason.
scoreable = df[df["label_kind"].isin(list(POSITIVE_KINDS) + ["unaffected"])].copy()
print("scoreable windows:", len(scoreable))
if scoreable["pair_id"].nunique() < MIN_RUNS:
    raise SystemExit(
        "Only %d pairs; a learned detector needs >= %d independent runs. "
        "Report the transparent baselines instead."
        % (scoreable["pair_id"].nunique(), MIN_RUNS))

## Split policy

Windows inside one run are autocorrelated: a fault spans many windows, so a window-level
split leaks. Splitting is therefore **by pair** (healthy + faulty twins of one workload stay
together), which is also the unit `evaluate.py` bootstraps over.

In [ ]:
pairs = sorted(scoreable["pair_id"].unique())
rng = np.random.RandomState(SEED)
shuffled = list(pairs)
rng.shuffle(shuffled)
n_test = max(1, len(shuffled) // 3)
test_pairs = set(shuffled[:n_test])
train_pairs = set(shuffled[n_test:])
print("train pairs:", len(train_pairs), "test pairs:", len(test_pairs))

train = scoreable[scoreable["pair_id"].isin(train_pairs)]
test = scoreable[scoreable["pair_id"].isin(test_pairs)]
y_train = (train["label_kind"].isin(POSITIVE_KINDS)).astype(int).values
y_test = (test["label_kind"].isin(POSITIVE_KINDS)).astype(int).values
print("train positives:", int(y_train.sum()), "/", len(y_train))
print("test positives:", int(y_test.sum()), "/", len(y_test))

In [ ]:
# Feature matrix: exactly the columns evaluate.py already validated.
X_train = train[FEATURE_COLUMNS].astype(float).values
X_test = test[FEATURE_COLUMNS].astype(float).values

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
except ImportError:
    raise SystemExit("scikit-learn is required for the learned baseline")

if len(set(y_train)) < 2:
    raise SystemExit(
        "The train split has only one class, so nothing can be learned. "
        "Report the transparent baselines; do not tune on the test split.")

scaler = StandardScaler().fit(X_train)
clf = LogisticRegression(max_iter=2000, random_state=SEED)
clf.fit(scaler.transform(X_train), y_train)
pred = clf.predict(scaler.transform(X_test))

tp, fp, fn, tn = prf_counts(y_test, pred)
precision, recall, f1, fpr = prf_from_counts(tp, fp, fn, tn)
print("logistic  P %.3f  R %.3f  F1 %.3f  FPR %.3f  (TP/FP/FN/TN %d/%d/%d/%d)"
      % (precision, recall, f1, fpr, tp, fp, fn, tn))

In [ ]:
# Run-level bootstrap CI over the TEST pairs (never over windows).
pred_by_run, true_by_run = {}, {}
for pair_id, p, t in zip(test["pair_id"].values, pred, y_test):
    pred_by_run.setdefault(pair_id, []).append(int(p))
    true_by_run.setdefault(pair_id, []).append(int(t))
pred_by_run = {k: np.array(v) for k, v in pred_by_run.items()}
true_by_run = {k: np.array(v) for k, v in true_by_run.items()}
ci = bootstrap_over_runs(pred_by_run, true_by_run, seed=SEED)
if ci is None:
    print("fewer than 2 test runs: no bootstrap CI, report the point estimate only")
else:
    (p_lo, p_hi), (r_lo, r_hi), (f_lo, f_hi), (fp_lo, fp_hi) = ci
    print("P  [%.2f, %.2f]" % (p_lo, p_hi))
    print("R  [%.2f, %.2f]" % (r_lo, r_hi))
    print("F1 [%.2f, %.2f]" % (f_lo, f_hi))
    print("FPR[%.2f, %.2f]" % (fp_lo, fp_hi))
print("test pairs:", len(true_by_run))

In [ ]:
# Transparent baselines computed by evaluate.py on the same dataset, for comparison.
if os.path.exists(metrics_path):
    metrics = pd.read_csv(metrics_path)
    print(metrics.to_string(index=False))
else:
    print("metrics.csv missing; re-run evaluate.py on the same data root")

## How to read this

- The learned model is only worth reporting if it beats the transparent baselines
  (robust threshold / residual threshold / EWMA / CUSUM) on the **same** test pairs.
- With a handful of runs the CI is wide by construction; a point estimate without it is
  not a result.
- These labels come from injected synthetic faults. A high score here means the pipeline can
  separate injected slowdowns from workload skew at this load and noise level. It says nothing
  about detecting real hardware degradation.
- Never tune on the test split. If the model looks bad, that is a finding, not a bug to hide.